In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import math
import os
import random

# Reproducability Code
def seed_everything(seed=1234):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(True)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- CONFIGURATION ---
SEQ_LEN = 50
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 1e-3

def create_sequences(df, feature_cols, seq_len):
    X, y = [], []
    for engine_id, group in df.groupby('id'):
        data = group[feature_cols].values
        labels = group['RUL'].values
        
        # Slide window
        for i in range(len(data) - seq_len + 1):
            X.append(data[i:i + seq_len])
            y.append(labels[i + seq_len - 1])
            
    return np.array(X), np.array(y)

def create_dataloader(X, y, shuffle=True):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, worker_init_fn=seed_worker)

# --- MODEL DEFINITION ---
class RULTransformer(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, dropout=0.1, seq_len=30):
        super(RULTransformer, self).__init__()
        self.input_linear = nn.Linear(input_dim, d_model)
        
        # Initialize positional encoding with a normal distribution (mean=0, std=0.02) and make it a learnable parameter
        self.pos_encoder = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
        
    def forward(self, src):
        # src shape: [batch_size, seq_len, input_dim]
        src = self.input_linear(src) + self.pos_encoder
        output = self.transformer_encoder(src)
        # Take the output of the last time step for sequence classification/regression
        output = output[:, -1, :]
        return self.regressor(output).squeeze(1)

Using device: cuda


In [3]:
def train_and_evaluate(train_path, test_path, model_save_path, hparams=None):
    # 1. Handle hyperparameters (fallback to globals if not doing HOPT)
    if hparams is None:
        current_seq_len = SEQ_LEN
        current_lr = LEARNING_RATE
        current_num_layers = 2
    else:
        current_seq_len = hparams['seq_len']
        current_lr = hparams['lr']
        current_num_layers = hparams['num_layers']

    train_full_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    feature_cols = [c for c in train_full_df.columns if c not in ['id', 'cycle', 'RUL']]

    all_ids = train_full_df['id'].unique()
    rng = np.random.default_rng(seed=42)
    rng.shuffle(all_ids)
    split = int(len(all_ids) * 0.8)
    train_df = train_full_df[train_full_df['id'].isin(all_ids[:split])].copy()
    val_df   = train_full_df[train_full_df['id'].isin(all_ids[split:])].copy()
    test_df  = test_df.copy()

    X_train, y_train = create_sequences(train_df, feature_cols, current_seq_len)
    X_val,   y_val   = create_sequences(val_df,   feature_cols, current_seq_len)
    X_test,  y_test  = create_sequences(test_df,  feature_cols, current_seq_len)

    train_loader = create_dataloader(X_train, y_train, shuffle=True)
    val_loader   = create_dataloader(X_val,   y_val,   shuffle=False)
    test_loader  = create_dataloader(X_test,  y_test,  shuffle=False)

    model = RULTransformer(
        input_dim=len(feature_cols),
        num_layers=current_num_layers,
        seq_len=current_seq_len
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=current_lr)

    best_val_rmse = float('inf')
    final_train_rmse = 0

    for epoch in range(EPOCHS):
        model.train()
        # Accumulate sum of squared errors, then divide by N for true RMSE
        train_mse_sum = 0.0

        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            # loss.item() is mean MSE over batch; multiply back to get sum of squared errors
            train_mse_sum += loss.item() * batch_X.size(0)

        train_rmse = float(np.sqrt(train_mse_sum / len(train_loader.dataset)))

        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(device)
                preds = model(batch_X)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(batch_y.numpy())
        val_rmse = math.sqrt(mean_squared_error(val_targets, val_preds))

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            torch.save(model.state_dict(), model_save_path)
            final_train_rmse = train_rmse

        # Print metrics every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == EPOCHS - 1:
            print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

    model.load_state_dict(torch.load(model_save_path, weights_only=True))
    model.eval()

    test_preds, test_targets = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            preds = model(batch_X)
            test_preds.extend(preds.cpu().numpy())
            test_targets.extend(batch_y.numpy())
    test_rmse = math.sqrt(mean_squared_error(test_targets, test_preds))

    # Clear memory to prevent OOM across 108 loops
    import gc
    del model, optimizer, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return {'Train RMSE': final_train_rmse, 'Val RMSE': best_val_rmse, 'Test RMSE': test_rmse}

# Store results by seed and dataset
all_results = {}

In [16]:
BASE_DATA_DIR = 'sutd_50039_deep_learning/data/processed-nasa-data/data_cleaning_1'

In [4]:
# 0. Train & Test on `linear_rul_no_norm_0` with multiple seeds
import os

# BASE_DATA_DIR = '/kaggle/input/datasets/jessicalorainegan/deep-learning/data/processed-nasa-data/data_cleaning_1'
BASE_DATA_DIR = 'sutd_50039_deep_learning/data/processed-nasa-data/data_cleaning_1'

SEEDS = [42, 1234, 999]
dataset_name_0 = 'FD001_linear_rul_no_norm_0'
train_file_0 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/train_processed_rul_only_fd001.csv')
test_file_0 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/test_processed_rul_only_fd001.csv')

all_results[dataset_name_0] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_0} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd001_no_norm_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_0, test_file_0, model_save_path)
    all_results[dataset_name_0][f'seed_{seed}'] = metrics


Training FD001_linear_rul_no_norm_0 with SEED=42
Epoch   1/50 | Train RMSE: 85.8196 | Val RMSE: 57.4686
Epoch   5/50 | Train RMSE: 58.3034 | Val RMSE: 56.1124
Epoch  10/50 | Train RMSE: 58.3605 | Val RMSE: 56.2309
Epoch  15/50 | Train RMSE: 58.4691 | Val RMSE: 56.1921
Epoch  20/50 | Train RMSE: 58.4979 | Val RMSE: 56.1375
Epoch  25/50 | Train RMSE: 58.4708 | Val RMSE: 56.0922
Epoch  30/50 | Train RMSE: 58.4660 | Val RMSE: 56.2518
Epoch  35/50 | Train RMSE: 58.5484 | Val RMSE: 56.2516
Epoch  40/50 | Train RMSE: 58.4774 | Val RMSE: 56.2424
Epoch  45/50 | Train RMSE: 58.5245 | Val RMSE: 56.0325
Epoch  50/50 | Train RMSE: 58.4690 | Val RMSE: 56.1129

Training FD001_linear_rul_no_norm_0 with SEED=1234
Epoch   1/50 | Train RMSE: 81.4500 | Val RMSE: 56.0499
Epoch   5/50 | Train RMSE: 58.1587 | Val RMSE: 56.1268
Epoch  10/50 | Train RMSE: 58.1831 | Val RMSE: 56.0958
Epoch  15/50 | Train RMSE: 58.2229 | Val RMSE: 56.2559
Epoch  20/50 | Train RMSE: 58.1590 | Val RMSE: 56.0514
Epoch  25/50 | Tra

In [7]:
# 1. Train & Test on `linear_rul_1` with multiple seeds
dataset_name_1 = 'linear_rul_1'
train_file_1 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/train_processed_rul_only_fd001.csv')
test_file_1 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/test_processed_rul_only_fd001.csv')

all_results[dataset_name_1] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_1} with SEED={seed}")
    print(f"{'='*60}")

    model_save_path = f'best_transformer_fd001_norm_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_1, test_file_1, model_save_path)
    all_results[dataset_name_1][f'seed_{seed}'] = metrics


Training linear_rul_1 with SEED=42
Epoch   1/50 | Train RMSE: 85.5870 | Val RMSE: 47.9624
Epoch   5/50 | Train RMSE: 27.4784 | Val RMSE: 24.9112
Epoch  10/50 | Train RMSE: 22.6812 | Val RMSE: 27.8416
Epoch  15/50 | Train RMSE: 20.1474 | Val RMSE: 26.6528
Epoch  20/50 | Train RMSE: 16.3513 | Val RMSE: 27.2298
Epoch  25/50 | Train RMSE: 15.0444 | Val RMSE: 29.1604
Epoch  30/50 | Train RMSE: 13.5880 | Val RMSE: 29.5945
Epoch  35/50 | Train RMSE: 14.4424 | Val RMSE: 27.1754
Epoch  40/50 | Train RMSE: 11.4967 | Val RMSE: 28.0981
Epoch  45/50 | Train RMSE: 11.5893 | Val RMSE: 29.0711
Epoch  50/50 | Train RMSE: 10.9012 | Val RMSE: 28.9879

Training linear_rul_1 with SEED=1234
Epoch   1/50 | Train RMSE: 79.5448 | Val RMSE: 37.6384
Epoch   5/50 | Train RMSE: 26.2663 | Val RMSE: 25.3050
Epoch  10/50 | Train RMSE: 18.8959 | Val RMSE: 27.6916
Epoch  15/50 | Train RMSE: 15.1125 | Val RMSE: 26.4216
Epoch  20/50 | Train RMSE: 12.9102 | Val RMSE: 29.7432
Epoch  25/50 | Train RMSE: 11.9470 | Val RMSE:

In [8]:
# 2. Train & Test on `piecewise_rul_2` with multiple seeds
dataset_name_2 = 'FD001_piecewise_rul_2'
train_file_2 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/train_processed_rul_piecewise_125_fd001.csv')   ## CHANGE TO from 150 to 125
test_file_2 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/test_processed_rul_piecewise_125_fd001.csv')

all_results[dataset_name_2] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_2} with SEED={seed}")
    print(f"{'='*60}")

    model_save_path = f'best_transformer_fd001_piecewise_2_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_2, test_file_2, model_save_path)
    all_results[dataset_name_2][f'seed_{seed}'] = metrics


Training FD001_piecewise_rul_2 with SEED=42
Epoch   1/50 | Train RMSE: 68.4005 | Val RMSE: 30.7218
Epoch   5/50 | Train RMSE: 13.8914 | Val RMSE: 13.3732
Epoch  10/50 | Train RMSE: 12.0886 | Val RMSE: 13.8189
Epoch  15/50 | Train RMSE: 10.9544 | Val RMSE: 13.1439
Epoch  20/50 | Train RMSE: 10.4957 | Val RMSE: 13.4305
Epoch  25/50 | Train RMSE: 9.7991 | Val RMSE: 12.6830
Epoch  30/50 | Train RMSE: 9.3227 | Val RMSE: 13.6098
Epoch  35/50 | Train RMSE: 9.1984 | Val RMSE: 13.2979
Epoch  40/50 | Train RMSE: 8.9571 | Val RMSE: 12.7354
Epoch  45/50 | Train RMSE: 8.7751 | Val RMSE: 12.3175
Epoch  50/50 | Train RMSE: 8.5647 | Val RMSE: 14.1594

Training FD001_piecewise_rul_2 with SEED=1234
Epoch   1/50 | Train RMSE: 62.5192 | Val RMSE: 21.3208
Epoch   5/50 | Train RMSE: 12.8850 | Val RMSE: 12.4541
Epoch  10/50 | Train RMSE: 11.0015 | Val RMSE: 15.3147
Epoch  15/50 | Train RMSE: 9.3690 | Val RMSE: 15.0599
Epoch  20/50 | Train RMSE: 8.6764 | Val RMSE: 13.4625
Epoch  25/50 | Train RMSE: 8.2340 | 

In [11]:
# 3. Train & Test on 'low_variance_1' with multiple seeds
dataset_name_3 = 'FD001_low_variance_1'

# BASE_FE_DIR = '/kaggle/input/datasets/jessicalorainegan/deep-learning/data/processed-nasa-data/feature_engineering_2'
BASE_FE_DIR = 'sutd_50039_deep_learning/data/processed-nasa-data/feature_engineering_2'

train_file_3 = os.path.join(BASE_FE_DIR, 'low_variance_1/train_fd001_low_variance_1_125.csv')    ## CHANGE TO low_variance_1_125
test_file_3 = os.path.join(BASE_FE_DIR, 'low_variance_1/test_fd001_low_variance_1_125.csv')

all_results[dataset_name_3] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_3} with SEED={seed}")
    print(f"{'='*60}")

    model_save_path = f'best_transformer_fd001_low_variance_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_3, test_file_3, model_save_path)
    all_results[dataset_name_3][f'seed_{seed}'] = metrics


Training FD001_low_variance_1 with SEED=42
Epoch   1/50 | Train RMSE: 60.8731 | Val RMSE: 25.3343
Epoch   5/50 | Train RMSE: 13.5791 | Val RMSE: 12.3641
Epoch  10/50 | Train RMSE: 11.3344 | Val RMSE: 13.1548
Epoch  15/50 | Train RMSE: 9.3302 | Val RMSE: 13.6815
Epoch  20/50 | Train RMSE: 8.5235 | Val RMSE: 13.3762
Epoch  25/50 | Train RMSE: 8.0764 | Val RMSE: 12.5482
Epoch  30/50 | Train RMSE: 7.8456 | Val RMSE: 13.5220
Epoch  35/50 | Train RMSE: 7.3923 | Val RMSE: 13.5218
Epoch  40/50 | Train RMSE: 7.3270 | Val RMSE: 13.1151
Epoch  45/50 | Train RMSE: 7.1154 | Val RMSE: 12.7396
Epoch  50/50 | Train RMSE: 7.0967 | Val RMSE: 14.5699

Training FD001_low_variance_1 with SEED=1234
Epoch   1/50 | Train RMSE: 63.7295 | Val RMSE: 24.5414
Epoch   5/50 | Train RMSE: 14.6817 | Val RMSE: 12.9418
Epoch  10/50 | Train RMSE: 12.6298 | Val RMSE: 12.8311
Epoch  15/50 | Train RMSE: 10.3595 | Val RMSE: 15.4871
Epoch  20/50 | Train RMSE: 9.2092 | Val RMSE: 16.0101
Epoch  25/50 | Train RMSE: 8.8978 | Val

In [12]:
# Train and Test on 'FD001 Drop S14' with multiple seeds
dataset_name_7 = 'FD001_drop_s14'
train_file_7 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/train_fd001_drop_s14.csv')
test_file_7 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/test_fd001_drop_s14.csv') 

all_results[dataset_name_7] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_7} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd001_drop_s14_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_7, test_file_7, model_save_path)
    all_results[dataset_name_7][f'seed_{seed}'] = metrics


Training FD001_drop_s14 with SEED=42
Epoch   1/50 | Train RMSE: 63.5126 | Val RMSE: 23.9926
Epoch   5/50 | Train RMSE: 14.0060 | Val RMSE: 12.6810
Epoch  10/50 | Train RMSE: 11.9489 | Val RMSE: 14.5890
Epoch  15/50 | Train RMSE: 9.4793 | Val RMSE: 14.8927
Epoch  20/50 | Train RMSE: 8.8160 | Val RMSE: 14.7391
Epoch  25/50 | Train RMSE: 8.4991 | Val RMSE: 15.1789
Epoch  30/50 | Train RMSE: 7.9057 | Val RMSE: 14.7313
Epoch  35/50 | Train RMSE: 7.7466 | Val RMSE: 15.2559
Epoch  40/50 | Train RMSE: 7.6657 | Val RMSE: 14.3607
Epoch  45/50 | Train RMSE: 7.2870 | Val RMSE: 14.8598
Epoch  50/50 | Train RMSE: 7.4256 | Val RMSE: 14.4975

Training FD001_drop_s14 with SEED=1234
Epoch   1/50 | Train RMSE: 60.8244 | Val RMSE: 24.8743
Epoch   5/50 | Train RMSE: 13.5872 | Val RMSE: 12.7797
Epoch  10/50 | Train RMSE: 11.4765 | Val RMSE: 14.7952
Epoch  15/50 | Train RMSE: 9.6857 | Val RMSE: 15.6222
Epoch  20/50 | Train RMSE: 8.4276 | Val RMSE: 16.1229
Epoch  25/50 | Train RMSE: 8.7949 | Val RMSE: 15.520

In [13]:
# Train and Test on 'FD001 Drop S14 and S11' with multiple seeds
dataset_name_8 = 'FD001_drop_s14_s11'
train_file_8 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/train_fd001_drop_s14_s11.csv')
test_file_8 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/test_fd001_drop_s14_s11.csv') 

all_results[dataset_name_8] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_8} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd001_drop_s14_s11_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_8, test_file_8, model_save_path)
    all_results[dataset_name_8][f'seed_{seed}'] = metrics


Training FD001_drop_s14_s11 with SEED=42
Epoch   1/50 | Train RMSE: 60.1535 | Val RMSE: 18.7467
Epoch   5/50 | Train RMSE: 13.4574 | Val RMSE: 13.7985
Epoch  10/50 | Train RMSE: 10.9824 | Val RMSE: 12.7869
Epoch  15/50 | Train RMSE: 8.9732 | Val RMSE: 14.6632
Epoch  20/50 | Train RMSE: 8.1533 | Val RMSE: 14.2215
Epoch  25/50 | Train RMSE: 7.7348 | Val RMSE: 14.0833
Epoch  30/50 | Train RMSE: 7.2873 | Val RMSE: 13.9318
Epoch  35/50 | Train RMSE: 7.3275 | Val RMSE: 13.5152
Epoch  40/50 | Train RMSE: 7.2357 | Val RMSE: 13.9472
Epoch  45/50 | Train RMSE: 7.0841 | Val RMSE: 14.5478
Epoch  50/50 | Train RMSE: 6.9850 | Val RMSE: 13.6338

Training FD001_drop_s14_s11 with SEED=1234
Epoch   1/50 | Train RMSE: 57.4711 | Val RMSE: 19.3558
Epoch   5/50 | Train RMSE: 13.3464 | Val RMSE: 12.4955
Epoch  10/50 | Train RMSE: 10.8943 | Val RMSE: 16.7587
Epoch  15/50 | Train RMSE: 8.6157 | Val RMSE: 14.3730
Epoch  20/50 | Train RMSE: 7.8593 | Val RMSE: 16.7521
Epoch  25/50 | Train RMSE: 7.3917 | Val RMSE

In [14]:
# Train and Test on 'FD001 Drop S14 and S12' with multiple seeds
dataset_name_9 = 'FD001_drop_s14_s12'
train_file_9 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/train_fd001_drop_s14_s12.csv')
test_file_9 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/test_fd001_drop_s14_s12.csv') 

all_results[dataset_name_9] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_9} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd001_drop_s14_s12_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_9, test_file_9, model_save_path)
    all_results[dataset_name_9][f'seed_{seed}'] = metrics


Training FD001_drop_s14_s12 with SEED=42
Epoch   1/50 | Train RMSE: 60.2844 | Val RMSE: 20.4202
Epoch   5/50 | Train RMSE: 13.6831 | Val RMSE: 12.9027
Epoch  10/50 | Train RMSE: 11.0402 | Val RMSE: 13.1952
Epoch  15/50 | Train RMSE: 9.3439 | Val RMSE: 15.1344
Epoch  20/50 | Train RMSE: 8.3481 | Val RMSE: 14.8102
Epoch  25/50 | Train RMSE: 7.9227 | Val RMSE: 14.4167
Epoch  30/50 | Train RMSE: 7.4077 | Val RMSE: 14.1330
Epoch  35/50 | Train RMSE: 7.4631 | Val RMSE: 14.4731
Epoch  40/50 | Train RMSE: 7.3910 | Val RMSE: 14.3430
Epoch  45/50 | Train RMSE: 7.1914 | Val RMSE: 15.0169
Epoch  50/50 | Train RMSE: 6.9729 | Val RMSE: 15.1380

Training FD001_drop_s14_s12 with SEED=1234
Epoch   1/50 | Train RMSE: 57.9347 | Val RMSE: 19.4434
Epoch   5/50 | Train RMSE: 13.5183 | Val RMSE: 13.4978
Epoch  10/50 | Train RMSE: 10.8561 | Val RMSE: 16.6148
Epoch  15/50 | Train RMSE: 8.9430 | Val RMSE: 14.3123
Epoch  20/50 | Train RMSE: 8.3025 | Val RMSE: 13.7710
Epoch  25/50 | Train RMSE: 7.4469 | Val RMSE

In [17]:
# 4. Train & Test on `linear_rul_no_norm_0` with multiple seeds
dataset_name_4 = 'FD002_linear_rul_no_norm_0'
train_file_4 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/train_processed_rul_only_fd002.csv')
test_file_4 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/test_processed_rul_only_fd002.csv')

all_results[dataset_name_4] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_4} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_no_norm_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_4, test_file_4, model_save_path)
    all_results[dataset_name_4][f'seed_{seed}'] = metrics


Training FD002_linear_rul_no_norm_0 with SEED=42
Epoch   1/50 | Train RMSE: 71.7230 | Val RMSE: 53.8046
Epoch   5/50 | Train RMSE: 59.4228 | Val RMSE: 54.3320
Epoch  10/50 | Train RMSE: 59.4164 | Val RMSE: 54.1869
Epoch  15/50 | Train RMSE: 59.3544 | Val RMSE: 53.9108
Epoch  20/50 | Train RMSE: 59.4275 | Val RMSE: 54.0886
Epoch  25/50 | Train RMSE: 59.3598 | Val RMSE: 53.8620
Epoch  30/50 | Train RMSE: 59.4077 | Val RMSE: 53.9588
Epoch  35/50 | Train RMSE: 59.3938 | Val RMSE: 53.8497
Epoch  40/50 | Train RMSE: 59.3522 | Val RMSE: 53.8759
Epoch  45/50 | Train RMSE: 59.4228 | Val RMSE: 54.0492
Epoch  50/50 | Train RMSE: 59.3339 | Val RMSE: 53.9327

Training FD002_linear_rul_no_norm_0 with SEED=1234
Epoch   1/50 | Train RMSE: 69.1612 | Val RMSE: 54.2477
Epoch   5/50 | Train RMSE: 59.1922 | Val RMSE: 54.0595
Epoch  10/50 | Train RMSE: 59.1328 | Val RMSE: 54.1616
Epoch  15/50 | Train RMSE: 59.1507 | Val RMSE: 53.9873
Epoch  20/50 | Train RMSE: 59.1081 | Val RMSE: 53.8279
Epoch  25/50 | Tra

In [19]:
# 5. Train & Test on `linear_rul_1` with multiple seeds
dataset_name_5 = 'FD002_linear_rul_1'
train_file_5 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/train_processed_rul_only_fd002.csv')
test_file_5 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/test_processed_rul_only_fd002.csv')

all_results[dataset_name_5] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_5} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd002_norm_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_5, test_file_5, model_save_path)
    all_results[dataset_name_5][f'seed_{seed}'] = metrics


Training FD002_linear_rul_1 with SEED=42
Epoch   1/50 | Train RMSE: 71.0340 | Val RMSE: 53.9139
Epoch   5/50 | Train RMSE: 59.2242 | Val RMSE: 54.5184
Epoch  10/50 | Train RMSE: 59.2539 | Val RMSE: 54.1217
Epoch  15/50 | Train RMSE: 59.1747 | Val RMSE: 53.9851
Epoch  20/50 | Train RMSE: 59.2459 | Val RMSE: 54.1152
Epoch  25/50 | Train RMSE: 59.2286 | Val RMSE: 53.8923
Epoch  30/50 | Train RMSE: 59.1864 | Val RMSE: 54.0557
Epoch  35/50 | Train RMSE: 59.2442 | Val RMSE: 53.9074
Epoch  40/50 | Train RMSE: 59.1920 | Val RMSE: 53.9634
Epoch  45/50 | Train RMSE: 59.2560 | Val RMSE: 54.2141
Epoch  50/50 | Train RMSE: 59.1633 | Val RMSE: 54.1539

Training FD002_linear_rul_1 with SEED=1234
Epoch   1/50 | Train RMSE: 65.2927 | Val RMSE: 48.1989
Epoch   5/50 | Train RMSE: 39.2664 | Val RMSE: 41.5185
Epoch  10/50 | Train RMSE: 38.0223 | Val RMSE: 36.7032
Epoch  15/50 | Train RMSE: 33.5628 | Val RMSE: 33.5049
Epoch  20/50 | Train RMSE: 31.8775 | Val RMSE: 31.8951
Epoch  25/50 | Train RMSE: 30.9248

In [20]:
# 6. Train & Test on `piecewise_rul_2` with multiple seeds
dataset_name_6 = 'FD002_piecewise_rul_2'
train_file_6 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/train_processed_rul_piecewise_125_fd002.csv')  ## FD002 change from 150 to 125
test_file_6 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/test_processed_rul_piecewise_125_fd002.csv')

all_results[dataset_name_6] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_6} with SEED={seed}")
    print(f"{'='*60}")
    
    # FIX (bug 3): was 'best_transformer_norm_1_seed_{seed}.pth' — same name as dataset 5!
    model_save_path = f'best_transformer_fd002_piecewise_2_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_6, test_file_6, model_save_path)
    all_results[dataset_name_6][f'seed_{seed}'] = metrics


Training FD002_piecewise_rul_2 with SEED=42
Epoch   1/50 | Train RMSE: 52.0174 | Val RMSE: 34.5841
Epoch   5/50 | Train RMSE: 23.9601 | Val RMSE: 24.5364
Epoch  10/50 | Train RMSE: 21.9543 | Val RMSE: 26.3706
Epoch  15/50 | Train RMSE: 18.1695 | Val RMSE: 16.7041
Epoch  20/50 | Train RMSE: 17.4315 | Val RMSE: 17.4504
Epoch  25/50 | Train RMSE: 16.8140 | Val RMSE: 15.8416
Epoch  30/50 | Train RMSE: 16.3651 | Val RMSE: 17.2196
Epoch  35/50 | Train RMSE: 15.9462 | Val RMSE: 16.5131
Epoch  40/50 | Train RMSE: 15.2810 | Val RMSE: 16.1738
Epoch  45/50 | Train RMSE: 15.1062 | Val RMSE: 15.7925
Epoch  50/50 | Train RMSE: 14.5152 | Val RMSE: 15.9467

Training FD002_piecewise_rul_2 with SEED=1234
Epoch   1/50 | Train RMSE: 52.1235 | Val RMSE: 41.4153
Epoch   5/50 | Train RMSE: 25.4836 | Val RMSE: 26.7872
Epoch  10/50 | Train RMSE: 23.2609 | Val RMSE: 24.5244
Epoch  15/50 | Train RMSE: 20.0241 | Val RMSE: 20.1041
Epoch  20/50 | Train RMSE: 17.8501 | Val RMSE: 16.8410
Epoch  25/50 | Train RMSE: 1

In [22]:
# Display detailed results and compute averages
print(f"\n{'='*80}")
print("DETAILED RESULTS BY SEED")
print(f"{'='*80}\n")

for dataset_name, seed_results in all_results.items():
    print(f"\n### {dataset_name.upper()} ###")
    df_results = pd.DataFrame(seed_results).T
    print(df_results.to_string())
    
    print(f"\n--- MEAN METRICS ---")
    mean_metrics = df_results.mean()
    for metric_name, value in mean_metrics.items():
        print(f"{metric_name}: {value:.2f}")

print(f"\n{'='*80}")
print("COMPARISON TABLE - ALL DATASETS & SEEDS")
print(f"{'='*80}\n")

# Create a comprehensive comparison table
comparison_data = []
for dataset_name, seed_results in all_results.items():
    for seed_label, metrics in seed_results.items():
        seed_num = seed_label.split('_')[1]
        comparison_data.append({
            'Dataset': dataset_name,
            'Seed': seed_num,
            'Train RMSE': metrics['Train RMSE'],
            'Val RMSE': metrics['Val RMSE'],
            'Test RMSE': metrics['Test RMSE']
        })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print(f"\n{'='*80}")
print("AVERAGES ACROSS ALL SEEDS")
print(f"{'='*80}\n")

# Create summary table with averages
summary_data = []
for dataset_name in all_results.keys():
    seed_dfs = [pd.DataFrame([metrics]).T for metrics in all_results[dataset_name].values()]
    avg_df = pd.concat(seed_dfs, axis=1).mean(axis=1)
    summary_data.append({
        'Dataset': dataset_name,
        'Mean Train RMSE': avg_df['Train RMSE'],
        'Mean Val RMSE': avg_df['Val RMSE'],
        'Mean Test RMSE': avg_df['Test RMSE']
    })

summary_df = pd.DataFrame(summary_data)
print("\n=== SUMMARY TABLE ===")
print(summary_df.to_string(index=False))

# Save comparison results to CSV
print(f"\n{'='*80}")
print("SAVING RESULTS TO CSV")
print(f"{'='*80}\n")

# Save detailed results
comparison_df.to_csv('transformer_results_detailed.csv', index=False)
print("✓ Saved: transformer_results_detailed.csv")

# Save summary results
summary_df.to_csv('transformer_results_summary.csv', index=False)
print("✓ Saved: transformer_results_summary.csv")

print("\nFiles saved in the current working directory.")


DETAILED RESULTS BY SEED


### FD001_LINEAR_RUL_NO_NORM_0 ###
           Train RMSE   Val RMSE  Test RMSE
seed_42     58.547068  56.028988  61.725498
seed_1234   58.173948  56.029241  61.846312
seed_999    58.123347  56.028968  61.747157

--- MEAN METRICS ---
Train RMSE: 58.28
Val RMSE: 56.03
Test RMSE: 61.77

### LINEAR_RUL_1 ###
           Train RMSE   Val RMSE  Test RMSE
seed_42     26.313455  24.104373  33.965472
seed_1234   27.174893  23.788535  31.912236
seed_999    24.971986  22.266871  34.005675

--- MEAN METRICS ---
Train RMSE: 26.15
Val RMSE: 23.39
Test RMSE: 33.29

### FD001_PIECEWISE_RUL_2 ###
           Train RMSE   Val RMSE  Test RMSE
seed_42      8.775126  12.317464  14.435583
seed_1234   12.885049  12.454147  13.750484
seed_999    13.947605  12.763221  12.726027

--- MEAN METRICS ---
Train RMSE: 11.87
Val RMSE: 12.51
Test RMSE: 13.64

### FD001_LOW_VARIANCE_1 ###
           Train RMSE   Val RMSE  Test RMSE
seed_42     13.579105  12.364109  13.479034
seed_1234   12.6297

# Hyperparameter Optimisation for Piecewise and Low Variance Models

This section performs grid search hyperparameter tuning for the following variables:
- `seed`
- `SEQ_LEN`
- `LEARNING_RATE`
- `num_layers`

on the`low_variance_1_125` models.

In [ ]:
train_opt = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/train_processed_rul_piecewise_150_fd001.csv')
test_opt = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/test_processed_rul_piecewise_150_fd001.csv')

In [ ]:
import itertools
import shutil

# Define hyperparameter search space
SEEDS = [42, 1234, 999]
SEQ_LENS = [30, 50, 70]
LEARNING_RATES = [1e-4, 5e-4, 1e-3]
NUM_LAYERS = [1, 2, 4]

# Save original config to restore later
ORIG_SEQ_LEN = SEQ_LEN
ORIG_BATCH_SIZE = BATCH_SIZE
ORIG_LR = LEARNING_RATE
ORIG_EPOCHS = EPOCHS

# --- Grid search for piecewise model ---
print("\n=== HYPERPARAM OPTIMISATION: PIECEWISE MODEL ===")
piecewise_results = []
best_piecewise_val_rmse = float('inf')
best_piecewise_model_path = 'best_transformer_hopt_piecewise_FINAL.pth'
temp_path = 'temp_hopt.pth'

for seed, seq_len, lr, num_layers in itertools.product(SEEDS, SEQ_LENS, LEARNING_RATES, NUM_LAYERS):
    print(f"\n--- seed={seed}, seq_len={seq_len}, lr={lr}, num_layers={num_layers} ---")
    seed_everything(seed)
    metrics = train_and_evaluate(
        train_file_2, test_file_2, temp_path,
        hparams={'seq_len': seq_len, 'num_layers': num_layers, 'lr': lr}
    )
    # Keep only the single best model across the entire grid
    if metrics['Val RMSE'] < best_piecewise_val_rmse:
        best_piecewise_val_rmse = metrics['Val RMSE']
        shutil.copy(temp_path, best_piecewise_model_path)
        print(f"  ✓ New best piecewise model (Val RMSE: {best_piecewise_val_rmse:.4f}) → {best_piecewise_model_path}")
    if os.path.exists(temp_path):
        os.remove(temp_path)
    piecewise_results.append({
        'seed': seed, 'seq_len': seq_len, 'lr': lr, 'num_layers': num_layers,
        **metrics
    })

piecewise_df = pd.DataFrame(piecewise_results)
best_piecewise_row = piecewise_df.loc[piecewise_df['Val RMSE'].idxmin()]
print(f"\nBest piecewise config: seed={best_piecewise_row['seed']}, seq_len={best_piecewise_row['seq_len']}, lr={best_piecewise_row['lr']}, num_layers={best_piecewise_row['num_layers']}")
print(f"Best piecewise Val RMSE: {best_piecewise_val_rmse:.4f} → saved to '{best_piecewise_model_path}'")
print(piecewise_df.to_string())

# --- Grid search for low variance model ---
print("\n=== HYPERPARAM OPTIMISATION: LOW VARIANCE MODEL ===")
lowvar_results = []
best_lowvar_val_rmse = float('inf')
best_lowvar_model_path = 'best_transformer_hopt_lowvar_FINAL.pth'

for seed, seq_len, lr, num_layers in itertools.product(SEEDS, SEQ_LENS, LEARNING_RATES, NUM_LAYERS):
    print(f"\n--- seed={seed}, seq_len={seq_len}, lr={lr}, num_layers={num_layers} ---")
    seed_everything(seed)
    metrics = train_and_evaluate(
        train_file_3, test_file_3, temp_path,
        hparams={'seq_len': seq_len, 'num_layers': num_layers, 'lr': lr}
    )
    # Keep only the single best model across the entire grid
    if metrics['Val RMSE'] < best_lowvar_val_rmse:
        best_lowvar_val_rmse = metrics['Val RMSE']
        shutil.copy(temp_path, best_lowvar_model_path)
        print(f"  ✓ New best low variance model (Val RMSE: {best_lowvar_val_rmse:.4f}) → {best_lowvar_model_path}")
    if os.path.exists(temp_path):
        os.remove(temp_path)
    lowvar_results.append({
        'seed': seed, 'seq_len': seq_len, 'lr': lr, 'num_layers': num_layers,
        **metrics
    })

lowvar_df = pd.DataFrame(lowvar_results)
best_lowvar_row = lowvar_df.loc[lowvar_df['Val RMSE'].idxmin()]
print(f"\nBest low variance config: seed={best_lowvar_row['seed']}, seq_len={best_lowvar_row['seq_len']}, lr={best_lowvar_row['lr']}, num_layers={best_lowvar_row['num_layers']}")
print(f"Best low variance Val RMSE: {best_lowvar_val_rmse:.4f} → saved to '{best_lowvar_model_path}'")
print(lowvar_df.to_string())

# Restore original config
SEQ_LEN = ORIG_SEQ_LEN
LEARNING_RATE = ORIG_LR

# Save results tables
piecewise_df.to_csv('hopt_piecewise_results.csv', index=False)
lowvar_df.to_csv('hopt_lowvar_results.csv', index=False)
print("\n✓ Saved: hopt_piecewise_results.csv, hopt_lowvar_results.csv")
print(f"✓ Best models saved: '{best_piecewise_model_path}', '{best_lowvar_model_path}'")  